In [1]:
%cd /kaggle/working
!rm -rf NEUROSEG
!git clone --depth 1 https://github.com/mo-karbalaee/NEUROSEG.git
%cd NEUROSEG
!pip install -e .

/kaggle/working
Cloning into 'NEUROSEG'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 92 (delta 4), reused 46 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 2.77 MiB | 4.50 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/kaggle/working/NEUROSEG
Obtaining file:///kaggle/working/NEUROSEG
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.3/203.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 86

In [2]:
# H2 — cross-species transfer, meaningful version (reuses H1's working fine-tune head).
# Pretrain JEPA self-supervised on the non-mouse SOURCE (Drosophila + zebrafish), then
# fine-tune on the mouse TARGET at each labeled fraction: cross-species-pretrained init
# ("finetune") vs from-scratch ("supervised_baseline"). Leakage-free (source != target species).
# Attach BOTH Kaggle datasets: the Drosophila/zebrafish source and neuroseg-labeled (mouse).
!PYTORCH_ALLOC_CONF=expandable_segments:True python main.py \
    --mode train --H2 \
    --source-data /kaggle/input/neuroseg-drosophila-larvae \
    --target-data /kaggle/input/datasets/mokarbalaee/neuroseg-labeled/neurofinder.00.00/neurofinder.00.00 \
    --output /kaggle/working/output \
    --config config.yaml

# Faster alternative — skip the ~1h source pretraining and reuse an already-trained
# cross-species checkpoint (upload the .pt AND its .json sidecar as a Kaggle dataset):
# !PYTORCH_ALLOC_CONF=expandable_segments:True python main.py \
#     --mode train --H2 \
#     --pretrained-ckpt /kaggle/input/.../jepa_pretrained_h2_XXXXXXXX.pt \
#     --target-data /kaggle/input/datasets/mokarbalaee/neuroseg-labeled/neurofinder.00.00/neurofinder.00.00 \
#     --output /kaggle/working/output \
#     --config config.yaml

[H2] device=cuda | pretrain(source)=/kaggle/input/neuroseg-drosophila-larvae | target=/kaggle/input/datasets/mokarbalaee/neuroseg-labeled/neurofinder.00.00/neurofinder.00.00 | dstc=32 | augment=True
Traceback (most recent call last):
  File "/kaggle/working/NEUROSEG/main.py", line 4, in <module>
    main()
  File "/kaggle/working/NEUROSEG/src/neuroseg/__main__.py", line 173, in main
    run(
  File "/kaggle/working/NEUROSEG/src/neuroseg/pipeline.py", line 99, in run
    return app.invoke({
           ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/langgraph/pregel/main.py", line 3913, in invoke
    for chunk in self.stream(
                 ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/langgraph/pregel/main.py", line 2967, in stream
    for _ in runner.tick(
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/langgraph/pregel/_runner.py", line 207, in tick
    run_with_retry(
  File "/usr/local/lib/python3.12/dist-packages/langgraph/pre